# 102 Spark optimizations

The goal of this lab is to understand some of the optimization mechanisms of Spark.

- Scala
    - [Spark programming guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html)
    - [RDD APIs](https://spark.apache.org/docs/latest/api/scala/org/apache/spark/rdd/RDD.html)
    - [PairRDD APIs](https://spark.apache.org/docs/latest/api/scala/org/apache/spark/rdd/PairRDDFunctions.html)
- Python
    - [Spark programming guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html)
    - [All RDD APIs](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.html)

Use `Tab` for autocompletion, `Shift+Tab` for documentation.

In [82]:
from pyspark.sql import SparkSession

# .master("local[N]") <-- ask for N cores on the driver
spark = SparkSession.builder \
.master("local[4]") \
.appName("Local Spark") \
.config('spark.ui.port', '4040') \
  .getOrCreate()
sc = spark.sparkContext

sc

<SparkContext master=local appName=Local Spark>

## The weather dataset

Download the following ZIP files and unzip them inside the "datasets/big" folder of this repo (which is not committed).
- [weather-sample1.txt](https://big.csr.unibo.it/downloads/bigdata/weather-datasets-s1.zip) <-- start from this!
- [weather-sample10.txt](https://big.csr.unibo.it/downloads/bigdata/weather-datasets-s10.zip)
- [weather-full.txt](https://big.csr.unibo.it/downloads/bigdata/weather-datasets-full.zip)
  
The weather datasets are textual files with weather data from all over the world in year 2000 (collected from the [National Climatic Data Center](ftp://ftp.ncdc.noaa.gov/pub/data/noaa/) of the USA. The full one weighs 13GB, the other are samples of 10% (1.3GB) and 1% (130MB) respectively.
  - Sample row: 005733213099999**19580101**03004+51317+028783FM-12+017199999V0203201N00721004501CN0100001N9 **-0021**1-01391102681
  - The date in YYYYMMDD format is located at 0-based position 15-23
  - The temperatue in x10 Celsius degrees is located at 0-based positions 87-92

In the dataset folder you also have *weather-stations.csv*; it is a structured file with the description of weather stations collecting the weather data.

In [83]:
# WEATHER structure: (usaf,wban,year,month,day,airTemperature,airTemperatureQuality)
def parseWeather(row):
    usaf = row[4:10]
    wban = row[10:15]
    year = row[15:19]
    month = row[19:21]
    day = row[21:23]
    airTemperature = row[87:92]
    airTemperatureQuality = row[92]

    return (usaf,wban,year,month,day,int(airTemperature)/10,airTemperatureQuality == '1')

# STATION structure: (usaf,wban,city,country,state,latitude,longitude,elevation,date_begin,date_end) 
def parseStation(row):
    def getDouble(str):
        return 0 if len(str)==0 else float(str)
    
    columns = [ x.replace("\"","") for x in row.split(",") ]
    latitude = getDouble(columns[6])
    longitude = getDouble(columns[7])
    elevation = getDouble(columns[8])
    return (columns[0],columns[1],columns[2],columns[3],columns[4],latitude,longitude,elevation,columns[9],columns[10])  

In [84]:
rddWeather = sc.\
  textFile("/home/jovyan/work/datasets/big/weather-sample1.txt").\
  map(lambda x: parseWeather(x))
rddStation = sc.\
  textFile("/home/jovyan/work/datasets/weather-stations.csv").\
  map(lambda x: parseStation(x))

## 102-1 Simple job optimization

Optimize the two jobs (avg temperature and max temperature) by avoiding the repetition of the same computations and by enforcing a partitioning criteria.
- There are multiple methods to repartition an RDD: check the ```coalesce```, ```partitionBy```, and ```repartition``` methods on the documentation and choose the best one.
- Verify your persisted data in the web UI
- Verify the execution plan of your RDDs with ```rdd.toDebugString``` or on the web UI

https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.coalesce.html?highlight=coalesce#pyspark.RDD.coalesce
https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.partitionBy.html?highlight=partitionby#pyspark.RDD.partitionBy
https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.repartition.html?highlight=repartition#pyspark.RDD.repartition

In [39]:
rddTemp = rddWeather.\
    filter(lambda x: x[5]<999).\
    map(lambda x: (x[3], (x[5],1))).\
    partitionBy(8).\
    cache()  # Cache the RDD to avoid recomputation

In [40]:
rddTemp.partitioner #to know if ther is a partition

In [ ]:
# Average temperature for every month
rddTemp.\
  reduceByKey(lambda v1, v2: (v1[0]+v2[0], v1[1]+v2[1]),1).\
  mapValues(lambda v: round(v[0]/v[1],2)).\
  collect()

In [ ]:
# Maximum temperature for every month
rddTemp.\
  reduceByKey(lambda x, y: y if x<y else x).\
  collect()

## 102-2 RDD preparation

Check the five possibilities to prepare the Station RDD for subsequent processing and identify the best one.

In [20]:
num_partitions = 8

# [0] and [1] are the fields composing the key; [3] and [7] are country and elevation, respectively
rddS1 = rddStation.\
  keyBy(lambda x: x[0] + x[1]).\
  partitionBy(num_partitions).\
  cache().\
  map(lambda kv: (kv[0],(kv[1][3],kv[1][7])))
rddS2 = rddStation.\
  keyBy(lambda x: x[0] + x[1]).\
  map(lambda kv: (kv[0],(kv[1][3],kv[1][7]))).\
  cache().\
  partitionBy(num_partitions)
rddS3 = rddStation.\
  keyBy(lambda x: x[0] + x[1]).\
  partitionBy(num_partitions).\
  map(lambda kv: (kv[0],(kv[1][3],kv[1][7]))).\
  cache()
rddS4 = rddStation.\
  keyBy(lambda x: x[0] + x[1]).\
  map(lambda kv: (kv[0],(kv[1][3],kv[1][7]))).\
  partitionBy(num_partitions).\
  cache()
rddS5 = rddStation.\
  map(lambda x: (x[0] + x[1], (x[3],x[7]))).\
  partitionBy(num_partitions).\
  cache()

## 102-3 Joining RDDs

Define the join between rddWeather and rddStation and compute:
- The maximum temperature for every city
- The maximum temperature for every city in the UK: 
  - ```StationData.country == "UK"```
- Sort the results by descending temperature
  - ```map(lambda kv: (kv[1],kv[0]))``` to invert key with value and vice versa

Hints & considerations:
- Keep only temperature values <999
- Join syntax: ```rdd1.join(rdd2)```
  - Both RDDs should be structured as key-value RDDs with the same key: usaf + wban
- Consider partitioning and caching to optimize the join
  - [Scala only] Careful: it is not enough for the two RDDs to have the same number of partitions; they must have the same partitioner! To create a partitioning function, you must ```import org.apache.spark.HashPartitioner``` and then define ```p = new HashPartitioner(n)``` where ```n``` is the number of partitions to create.
- Verify the execution plan of the join in the web UI

In [ ]:
print("rddWeather:")
for row in rddWeather.take(3):
    print(row)

print("rddStation:")
for row in rddStation.take(3):
    print(row)

In [ ]:
#keeping only temperatures value < 999
rddTempFiltered = rddWeather.\
    filter(lambda x: x[5]<999).\
    cache()  
    
print("rddTempFiltered:")
for row in rddTempFiltered.take(3):
    print(row)

In [ ]:
numPartitions = 8

weatherKV = (
    rddWeather
    .filter(lambda x: x[5] < 999)
    .map(lambda x: (x[0] + x[1], x))
    .partitionBy(numPartitions)
    .cache()
)

stationKV = (
    rddStation
    .map(lambda x: (x[0] + x[1], x))
    .partitionBy(numPartitions)
    .cache()
)

print("weatherKV:")
for row in weatherKV.take(3):
    print(row)

print("stationKV:")
for row in stationKV.take(3):
    print(row)

weatherKV:
('02875099999', ('028750', '99999', '2000', '01', '01', -13.7, True))
('02875099999', ('028750', '99999', '2000', '01', '01', -13.0, True))
('02875099999', ('028750', '99999', '2000', '01', '01', -13.0, True))
stationKV:
('00702699999', ('007026', '99999', 'WXPOD 7026', 'AF', '', 0.0, 0.0, 7026.0, '20120713', '20170715'))
('00705999999', ('007059', '99999', 'CWOS 07059', '', '', 0, 0, 0, '20120314', '20120828'))
('00706499999', ('007064', '99999', 'CWOS 07064', '', '', 0, 0, 0, '20121218', '20121219'))


In [58]:
joinedRdd = weatherKV.join(stationKV).cache()

for row in joinedRdd.take(3):
    print(row)

('02875099999', (('028750', '99999', '2000', '01', '01', -13.7, True), ('028750', '99999', 'OULU', 'FI', '', 64.93, 25.355, 14.3, '19060101', '20170715')))
('02875099999', (('028750', '99999', '2000', '01', '01', -13.0, True), ('028750', '99999', 'OULU', 'FI', '', 64.93, 25.355, 14.3, '19060101', '20170715')))
('02875099999', (('028750', '99999', '2000', '01', '01', -13.0, True), ('028750', '99999', 'OULU', 'FI', '', 64.93, 25.355, 14.3, '19060101', '20170715')))


In [70]:
maxTempByCity = (
    joinedRdd
    .map(lambda kv: (kv[1][1][2], kv[1][0][5]))
    .reduceByKey(lambda x, y: max(x, y))
)
print(maxTempByCity.collect())

[('STORNOWAY', 20.0), ('LOCH GLASCARNOCH', 22.9), ('HAILUOTO ISLAND', 28.6), ('BARRA ISLAND', 21.0), ('KINLOSS', 24.0), ('SEINAJOKI', 28.0), ('WATERSTEIN', 21.0), ('SUOMUSJARVI', 26.8), ('BALTASOUND NO.2', 18.0), ('LAPPEENRANTA HIEKKAPAKKA', 23.1), ('VARKAUS', 29.0), ('KIRKWALL', 19.0), ('TULLOCH BRIDGE', 26.0), ('JOENSUU', 30.0), ('SUMBURGH', 30.0), ('HELSINKI MALMI', 28.0), ('ALTNAHARRA NO2', 24.1), ('LAHTI LAUNE', 29.5), ('ISOSAARI', 21.6), ('MUCKLE HOLM', 16.0), ('AHTARI MYLLYMAKI', 28.9), ('SELLA NESS', 19.1), ('FOULA NO2', 15.9), ('TAIN RANGE (SAWS)', 22.1), ('MUSTASAARI VALASSAARET', 20.2), ('SCATSTA', 20.0), ('VAASA', 27.0), ('LEMLAND NYHAMN', 19.8), ('INKOO BAGASKAR', 22.9), ('NIVALA', 28.7), ('HALLI', 29.1), ('FOULA', 15.9), ('KAUHAVA', 29.6), ('JOMALA', 22.2), ('FAIR ISLE', 15.7), ('SKYE/LUSA', 23.4), ('VIITASAARI', 28.9), ('SULE SKERRY', 17.4), ('LOSSIEMOUTH', 23.4), ('KOTKA RANKKI', 25.3), ('HELSINKI VANTAA', 28.1), ('KUMLINGE ISLAND', 22.8), ('AVIEMORE', 23.5), ('TAMPERE 

In [72]:
maxTempUK = (
    joinedRdd
    .filter(lambda city: city[1][1][3] == "UK")
    .map(lambda kv: (kv[1][1][2], kv[1][0][5]))
    .reduceByKey(lambda x, y: max(x, y))
)
print(maxTempUK.collect())

[('STORNOWAY', 20.0), ('LOCH GLASCARNOCH', 22.9), ('BARRA ISLAND', 21.0), ('KINLOSS', 24.0), ('WATERSTEIN', 21.0), ('BALTASOUND NO.2', 18.0), ('KIRKWALL', 19.0), ('TULLOCH BRIDGE', 26.0), ('SUMBURGH', 30.0), ('ALTNAHARRA NO2', 24.1), ('MUCKLE HOLM', 16.0), ('SELLA NESS', 19.1), ('FOULA NO2', 15.9), ('TAIN RANGE (SAWS)', 22.1), ('SCATSTA', 20.0), ('FOULA', 15.9), ('FAIR ISLE', 15.7), ('SKYE/LUSA', 23.4), ('SULE SKERRY', 17.4), ('LOSSIEMOUTH', 23.4), ('AVIEMORE', 23.5), ('BUTT OF LEWIS (LH)', 8.8), ('AONACH MOR', 17.4), ('NORTH RONA ISLAND', 21.7), ('BENBECULA', 20.0), ('GLENLIVET', 23.0), ('LERWICK', 17.9), ('NORTH RONALDSAY ISL', 6.9), ('SOUTH UIST RANGE', 34.2), ('CAIRNGORM SUMMIT', 14.9), ('FOYERS', 23.5), ('LOCHBOISDALE', 21.0), ('INVERNESS', 23.0), ('AULTBEA NO2', 23.0), ('INVERGORDON HARBOUR', 20.8), ('RACKWICK', 19.6), ('KILMORY', 21.2)]


In [ ]:
maxTempByCitySorted = (
    maxTempByCity
    .map(lambda kv: (kv[1], kv[0]))
    .sortByKey(ascending = False)
)

maxTempUKSorted = (
    maxTempUK
    .map(lambda kv: (kv[1], kv[0]))
    .sortByKey(False)
)

print(maxTempByCitySorted.collect())
print(maxTempUKSorted.collect())

[(34.2, 'SOUTH UIST RANGE'), (31.4, 'OULU'), (30.0, 'JOENSUU'), (30.0, 'SUMBURGH'), (30.0, 'KUOPIO'), (30.0, 'SAVONLINNA'), (30.0, 'KAJAANI'), (29.8, 'TAMPERE PIRKKALA'), (29.6, 'KAUHAVA'), (29.5, 'LAHTI LAUNE'), (29.4, 'UTTI'), (29.4, 'KANKAANPAA NIINISALO PUOLVOIM'), (29.2, 'SUOMUSSALMI'), (29.1, 'HALLI'), (29.0, 'VARKAUS'), (29.0, 'KUUSAMO'), (29.0, 'JYVASKYLA'), (29.0, 'LAPPEENRANTA'), (29.0, 'PORI'), (28.9, 'AHTARI MYLLYMAKI'), (28.9, 'VIITASAARI'), (28.7, 'NIVALA'), (28.6, 'HAILUOTO ISLAND'), (28.1, 'HELSINKI VANTAA'), (28.0, 'SEINAJOKI'), (28.0, 'HELSINKI MALMI'), (28.0, 'MIKKELI'), (28.0, 'KRUUNUPYY'), (27.1, 'TURKU'), (27.0, 'VAASA'), (27.0, 'JOKIOINEN'), (26.8, 'SUOMUSJARVI'), (26.0, 'TULLOCH BRIDGE'), (25.3, 'KOTKA RANKKI'), (24.1, 'ALTNAHARRA NO2'), (24.0, 'KINLOSS'), (23.5, 'AVIEMORE'), (23.5, 'FOYERS'), (23.4, 'SKYE/LUSA'), (23.4, 'LOSSIEMOUTH'), (23.1, 'LAPPEENRANTA HIEKKAPAKKA'), (23.0, 'GLENLIVET'), (23.0, 'INVERNESS'), (23.0, 'AULTBEA NO2'), (23.0, 'MARIEHAMN'), (22.9

In [86]:
# prof solution
rddW = (
    rddWeather
    .filter(lambda x: x[6]) 
    .map(lambda x: (x[0] + x[1], x[5]))
)

rddS = (
    rddStation
    .map(lambda x: (x[0] + x[1], (x[2], x[3])))
)

# first solution
#rddJ = rddW.join(rddS).cache() (stationId, (temp, (city, country)))
# rddJ.\
#     map(lambda j: (j[1][1][0], j[1][0])).\
#     reduceByKey(lambda t1, t2: max(t1, t2)).\
#     collect()
    
# rddJ.\
#     filter(lambda j: j[1][1][1] == "UK").\
#     map(lambda j: (j[1][1][0], j[1][0])).\
#     reduceByKey(lambda t1, t2: max(t1, t2)).\
#     collect()
    
# rddJ.\
#     filter(lambda j: j[1][1][1] == "UK").\
#     map(lambda j: (j[1][1][0], j[1][0])).\
#     reduceByKey(lambda t1, t2: max(t1, t2)).\
#     map(lambda kv: (kv[1], kv[0])).\
#     sortByKey(False).\
#     collect()

# second solution
rddJ = rddW.\
    join(rddS).\
    map(lambda j: (j[1][1], j[1][0])).\
    reduceByKey(lambda t1, t2: max(t1, t2)).\
    cache() #((city, country), maxTemp)

rddJ.\
    map(lambda j: (j[0][0], j[1])).\
    collect()
    
rddJ.\
    filter(lambda j: j[0][1] == "UK").\
    map(lambda j: (j[0][0], j[1])).\
    collect()
    
rddJ.\
    filter(lambda j: j[0][1] == "UK").\
    map(lambda j: (j[1], j[0][0])).\
    sortByKey(False).\
    collect()

[(34.2, 'SOUTH UIST RANGE'),
 (30.0, 'SUMBURGH'),
 (26.0, 'TULLOCH BRIDGE'),
 (24.1, 'ALTNAHARRA NO2'),
 (24.0, 'KINLOSS'),
 (23.5, 'AVIEMORE'),
 (23.5, 'FOYERS'),
 (23.4, 'SKYE/LUSA'),
 (23.4, 'LOSSIEMOUTH'),
 (23.0, 'GLENLIVET'),
 (23.0, 'INVERNESS'),
 (23.0, 'AULTBEA NO2'),
 (22.9, 'LOCH GLASCARNOCH'),
 (22.1, 'TAIN RANGE (SAWS)'),
 (21.7, 'NORTH RONA ISLAND'),
 (21.2, 'KILMORY'),
 (21.0, 'WATERSTEIN'),
 (21.0, 'LOCHBOISDALE'),
 (21.0, 'BARRA ISLAND'),
 (20.8, 'INVERGORDON HARBOUR'),
 (20.0, 'BENBECULA'),
 (20.0, 'STORNOWAY'),
 (20.0, 'SCATSTA'),
 (19.6, 'RACKWICK'),
 (19.1, 'SELLA NESS'),
 (19.0, 'KIRKWALL'),
 (18.0, 'BALTASOUND NO.2'),
 (17.9, 'LERWICK'),
 (17.4, 'AONACH MOR'),
 (17.4, 'SULE SKERRY'),
 (16.0, 'MUCKLE HOLM'),
 (15.9, 'FOULA NO2'),
 (15.9, 'FOULA'),
 (15.7, 'FAIR ISLE'),
 (14.9, 'CAIRNGORM SUMMIT'),
 (8.8, 'BUTT OF LEWIS (LH)'),
 (6.9, 'NORTH RONALDSAY ISL')]

## 102-4 Memory occupation

Use Spark's web UI to verify the space occupied by the provided RDDs.

*Warning*: in PySpark, StoraleLevels use serialization by default (see [documentation](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.StorageLevel.html)).

In [11]:
from pyspark import StorageLevel

# Clear the cache
for (id, rdd) in sc._jsc.getPersistentRDDs().items():         
    rdd.unpersist()

memSerRdd = rddWeather.cache()
memRdd = memSerRdd.map(lambda x: x).persist(StorageLevel. MEMORY_AND_DISK_DESER)
diskRdd = memSerRdd.map(lambda x: x).persist(StorageLevel.DISK_ONLY)

## 102-5 Evaluating different join methods

Consider the following scenario:
- We have a disposable RDD of Weather data (i.e., it is used only once): ```rddW```
- And we have an RDD of Station data that is used many times: ```rddS```
- Both RDDs are cached (```collect()```is called to enforce caching)

We want to join the two RDDS. Which option is best?
- Simply join the two RDDs
- Enforce on ```rddW1``` the same partitioner of ```rddS``` (and then join)
- Exploit broadcast variables

In [ ]:
num_partitions = 8

rddW = rddWeather.\
  filter(lambda w: w[5]<999).\
  keyBy(lambda w: w[0]+w[1]).\
  cache()

rddS = rddStation.\
  keyBy(lambda s: s[0]+s[1]).\
  partitionBy(num_partitions).\
  cache()

# Collect to enforce caching
rddW.collect()
rddS.collect()

In [ ]:
# Is it better to simply join the two RDDs..
rddX = rddW.\
  join(rddS).\
  map(lambda kv: (kv[1][1][2],kv[1][0][5])).\
  reduceByKey(lambda x,y: min(x,y),1)
print(rddX.toDebugString().decode("unicode_escape"))

In [ ]:
rddX.collect()

In [ ]:
# ..to enforce on rddW1 the same partitioner of rddS..
rddX = rddW.\
  partitionBy(num_partitions).\
  join(rddS).\
  map(lambda kv: (kv[1][1][2],kv[1][0][5])).\
  reduceByKey(lambda x,y: min(x,y),1)
print(rddX.toDebugString().decode("unicode_escape"))

In [ ]:
rddX.collect()

In [ ]:
# ..or to exploit broadcast variables?
bRddS = sc.broadcast(rddS.map(lambda s: (s[0], s[1][2])).collectAsMap())
rddJ = rddW.\
  map(lambda kv: (bRddS.value.get(kv[0]), kv[1][5])).\
  filter(lambda x: x[0] is not None)

rddX = rddJ.\
  reduceByKey(lambda x,y: min(x,y),1)
print(rddX.toDebugString().decode("unicode_escape"))

In [ ]:
rddX.collect()